[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch09-data-selection/01-gia-tri-cua-mot-mau.ipynb)

# Giá trị của một mẫu: ICR, lợi ích giảm dần và nhãn nhiễu

Chương 9 mở đầu bằng một nhận xét đơn giản: huấn luyện trả cùng một lượt xuôi và lượt ngược cho
mọi mẫu, nhưng các mẫu không đem lại cùng một phần học. Chương gom nhận xét đó vào một tỉ số,
tỉ số thông tin trên tính toán, hay ICR (information-compute ratio): mỗi FLOP huấn luyện đem
lại bao nhiêu phần học.

Sổ tay này tính lại các phép tính nhẩm của chương về ICR và về chất lượng nhãn, rồi đo chính
những đại lượng đó trên một tập dữ liệu thật, FashionMNIST, để bạn thấy hình dạng mà chương mô tả
xuất hiện trên máy của mình.

**Bạn sẽ làm:**
1. đọc lại kịch bản 70 tỷ tham số và lý do ba tầng tối ưu nhân với nhau;
2. tính ICR của napkin math 1.1 cho chọn ngẫu nhiên và cho một tập lõi (coreset) 50 phần trăm,
   rồi thêm phụ trội chọn lọc mà phép tính của sách để ra ngoài;
3. đo một đường cong học tập trên FashionMNIST và đọc nó thành ICR biên và PPD;
4. tính hệ số chất lượng dữ liệu của napkin math 1.2, rồi cấy nhiễu nhãn vào dữ liệu thật: cần
   bao nhiêu mẫu nhiễu để bằng mẫu sạch, và một mô hình thay thế (proxy model) có tìm lại được
   các nhãn bị lật không.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Tường dữ liệu: kịch bản 70 tỷ tham số

Chương dựng một kịch bản: một mô hình ngôn ngữ 70 tỷ tham số, ngân sách 10,000 GPU H100 trong 3
tháng. Ở mức sử dụng FLOP 40 phần trăm, cụm ấy xử lý được khoảng 73.2 nghìn tỷ token. Đội chỉ có
một kho 5 nghìn tỷ token đã lọc, sẵn sàng dùng. Câu hỏi đầu tiên: ngân sách đó đọc hết kho bao
nhiêu lần?

In [ ]:
NGHIN_TY = 1e12
TOKEN_XU_LY = sach(73.2, nguon="≈73.2 nghìn tỷ token · 40 phần trăm", tol=0.05) * NGHIN_TY
KHO_DA_LOC = sach(5, nguon="5 nghìn tỷ token") * NGHIN_TY
so_lan = TOKEN_XU_LY / KHO_DA_LOC
theo_sach("số lần cụm đọc hết kho đã lọc", so_lan, sach=14.6, nguon="5 nghìn tỷ token · ≈14.6×", tol=0.05)

# Giá của ngân sách ấy. Chương không ghi đơn giá; suy ngược ra nó, với giả định 3 tháng = 90 ngày.
SO_GPU = sach(10_000, nguon="10,000 H100 · 3 tháng")
THANG = sach(3, nguon="10,000 H100 · 3 tháng")
GIA = sach(86.4e6, nguon="≈86.4 triệu đô la", tol=0.05e6)
gio_gpu = SO_GPU * THANG * 30 * 24
print(f"{SO_GPU:,} GPU × {THANG} tháng × 30 ngày × 24 giờ = {gio_gpu:,} giờ GPU")
print(f"đơn giá suy ra: {GIA:,.0f} đô la / {gio_gpu:,} giờ = {GIA / gio_gpu:.2f} đô la mỗi giờ GPU")
print(f"mỗi GPU phải xử lý {TOKEN_XU_LY / (gio_gpu * 3600):,.0f} token mỗi giây, suốt 3 tháng")

Đơn giá suy ra trùng với mốc 4 đô la mỗi giờ GPU mà bảng 15 của chương dùng ở phần mô hình chi phí.
Con số quan trọng là 14.6: tính toán dư gấp nhiều lần dữ liệu tốt. Khi đó, chương nói, đội có ba lựa
chọn: lặp nhiều epoch, hạ ngưỡng chất lượng, hoặc đầu tư vào chọn lọc để rút nhiều phần học hơn từ
mỗi token.

### Vì sao ba tầng tối ưu nhân với nhau

Chọn lọc dữ liệu giảm số mẫu, tầng mô hình giảm số phép toán mỗi mẫu, tầng máy tăng thông lượng
(throughput) hiệu dụng. Thời gian huấn luyện tỉ lệ với **tích** của số mẫu và phép toán mỗi mẫu,
chia cho thông lượng. Mỗi tầng chia phần việc còn lại, chứ không trừ đi một phần cố định.

**Dự đoán:** giảm 2 lần ở cả ba tầng thì tổng thời gian giảm bao nhiêu lần?

In [ ]:
GIAM = sach(2, nguon="2× · 2× · 2× = 8×, không phải 6×")
so_mau, phep_moi_mau, thong_luong = 1.0, 1.0, 1.0  # chuẩn hoá về mốc ban đầu
thoi_gian = lambda d, o, r: d * o / r

goc = thoi_gian(so_mau, phep_moi_mau, thong_luong)
sau = thoi_gian(so_mau / GIAM, phep_moi_mau / GIAM, thong_luong * GIAM)
theo_sach("mức giảm khi ba tầng cùng tác động", goc / sau, sach=8, nguon="2× · 2× · 2× = 8×, không phải 6×")
print(f"cộng ba mức giảm, cách tính sai: {GIAM + GIAM + GIAM}×")

TIEN = sach(100e6, nguon="50 phần trăm · 100 triệu đô la · 50 triệu đô la")
theo_sach("tiết kiệm khi giảm nửa tập, lịch cố định", 0.5 * TIEN, sach=50e6,
          nguon="50 phần trăm · 100 triệu đô la · 50 triệu đô la")

Cả hai phép tính đều có điều kiện mà chương nói rõ: epoch, batch và phần việc mỗi mẫu giữ nguyên, và
cả ba tầng tác động lên **cùng một** điểm nghẽn. Nếu lịch huấn luyện thêm bước để bù cho tập nhỏ hơn,
phần tiết kiệm co lại.

## 2. Napkin math 1.1: ICR của một tập lõi

Chương định nghĩa $\text{ICR} = \Delta I / \Delta\text{FLOPs}$. Tử số không đo trực tiếp được, nên phép
tính dùng một chỉ số gián tiếp: mức tăng độ chính xác. Kịch bản là ResNet-50 trên ImageNet, một epoch,
với mức tăng **giả định**; sách nói rõ đây không phải kết quả EL2N đã báo cáo.

In [ ]:
N_IMAGENET = sach(1.28e6, nguon="1.28 triệu ảnh")
GFLOP = 1e9
FLOP_XUOI = sach(8.2, nguon="1.28 triệu ảnh · ~8.2 GFLOP · ~24.6 GFLOP", tol=0.05) * GFLOP
FLOP_BUOC = sach(24.6, nguon="1.28 triệu ảnh · ~8.2 GFLOP · ~24.6 GFLOP", tol=0.05) * GFLOP
print(f"một bước huấn luyện = {FLOP_BUOC / FLOP_XUOI:.1f} lần một lượt xuôi")

C_epoch = N_IMAGENET * FLOP_BUOC
theo_sach("FLOP một epoch", C_epoch, sach=3.15e16, nguon="3.15 · 10¹⁶ FLOP")

TANG_NGAU_NHIEN = sach(5, nguon="5 điểm phần trăm · 1.6 · 10⁻¹⁶")
icr_ngau_nhien = TANG_NGAU_NHIEN / C_epoch
theo_sach("ICR chọn ngẫu nhiên (điểm mỗi FLOP)", icr_ngau_nhien, sach=1.6e-16, nguon="5 điểm phần trăm · 1.6 · 10⁻¹⁶")

# Tập lõi EL2N giữ 50 phần trăm dữ liệu, với mức tăng giả định 4.5 điểm.
GIU = sach(50, nguon="50 phần trăm · 640.6 nghìn") / 100
TANG_TAP_LOI = sach(4.5, nguon="4.5 điểm (90 phần trăm)")
theo_sach("phần mức tăng tập lõi giữ lại (%)", 100 * TANG_TAP_LOI / TANG_NGAU_NHIEN, sach=90, nguon="4.5 điểm (90 phần trăm)")
C_tap_loi = GIU * N_IMAGENET * FLOP_BUOC
theo_sach("FLOP của tập lõi", C_tap_loi, sach=1.6e16, nguon="1.6 · 10¹⁶ FLOP")
icr_tap_loi = TANG_TAP_LOI / C_tap_loi
theo_sach("ICR tập lõi (điểm mỗi FLOP)", icr_tap_loi, sach=2.9e-16, nguon="2.9 · 10⁻¹⁶")
theo_sach("ICR tập lõi so với chọn ngẫu nhiên", icr_tap_loi / icr_ngau_nhien, sach=1.8, nguon="1.8× · 0.5 điểm phần trăm")
theo_sach("mức tăng thấp hơn (điểm)", TANG_NGAU_NHIEN - TANG_TAP_LOI, sach=0.5, nguon="1.8× · 0.5 điểm phần trăm")

In [ ]:
SO_ANH_SACH = sach(640.6e3, nguon="50 phần trăm · 640.6 nghìn")
print(f"50 phần trăm của 1.28 triệu ảnh: {GIU * N_IMAGENET:,.0f} ảnh · sách in: {SO_ANH_SACH:,.0f} ảnh")
print(f"nhân ngược con số của sách: 2 × {SO_ANH_SACH:,.0f} = {2 * SO_ANH_SACH:,.0f} ảnh")

Phép chia đôi không khớp ở chữ số cuối, và đó không phải lỗi của bạn: 640.6 nghìn là một nửa của khoảng
1.2812 triệu ảnh, tức số ảnh trước khi làm tròn thành 1.28 triệu. Sách tính trên số chưa làm tròn rồi
mới in ra từng con số. Các kết quả 1.6, 2.9 và 1.8 không đổi ở độ chính xác sách in.

Để ý rằng tỉ số 1.8 không phụ thuộc vào ResNet-50 hay ImageNet. Nó chỉ là phần mức tăng giữ lại chia
cho phần dữ liệu giữ lại, $0.9 / 0.5$. Tập lõi có ICR cao hơn chọn ngẫu nhiên khi và chỉ khi nó giữ
được phần mức tăng lớn hơn phần dữ liệu.

### Phần sách để ra ngoài: phụ trội chọn lọc

Sách nói phép tính "không tính phụ trội chọn lọc". Chấm điểm EL2N cần huấn luyện một mô hình thay thế
vài epoch, và lượt chấm ấy cũng tốn FLOP. Gọi phụ trội là $s$, đo bằng số epoch của cả tập. ICR của tập
lõi trở thành $\Delta I / \big((f + s)\,C_{\text{epoch}}\big)$ với $f$ là phần dữ liệu giữ lại.

**Dự đoán:** với các giả định của sách ($f = 0.5$, giữ 90 phần trăm mức tăng), phụ trội được phép lớn
tới đâu trước khi tập lõi mất lợi thế so với chọn ngẫu nhiên?

In [ ]:
def ti_so_icr(giu, phan_tang, phu_troi):
    """ICR của tập lõi chia cho ICR chọn ngẫu nhiên; phụ trội tính bằng epoch của cả tập."""
    return phan_tang / (giu + phu_troi)


phan_tang_sach = TANG_TAP_LOI / TANG_NGAU_NHIEN
hoa_von = phan_tang_sach - GIU  # ti_so_icr = 1 khi phụ trội bằng hiệu này
print(f"tập lõi còn lợi khi phụ trội dưới {hoa_von:.1f} epoch của cả tập")
assert abs(ti_so_icr(GIU, phan_tang_sach, hoa_von) - 1) < 1e-12

s = np.linspace(0, 0.8, 81)
fig, ax = plt.subplots(figsize=(7.5, 4))
for phan_tang, mau in ((0.9, "C0"), (0.7, "C1"), (0.55, "C2")):
    nhan = f"giữ {phan_tang:.0%} mức tăng" + (" (giả định của sách)" if phan_tang == 0.9 else "")
    ax.plot(s, ti_so_icr(GIU, phan_tang, s), color=mau, lw=2, label=nhan)
ax.axhline(1, color="0.4", lw=1, ls="--")
ax.text(0.79, 1.03, "ngang chọn ngẫu nhiên", ha="right", fontsize=8, color="0.3")
ax.plot([0], [ti_so_icr(GIU, 0.9, 0)], "o", color="C0", ms=8)
ax.annotate("sách: 1.8×, chưa tính phụ trội", (0, 1.8), xytext=(25, 4), textcoords="offset points", fontsize=8,
            arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.plot([hoa_von], [1], "o", mfc="white", mec="C0", mew=2, ms=8)
ax.annotate(f"hoà vốn: {hoa_von:.1f} epoch", (hoa_von, 1), xytext=(8, -26), textcoords="offset points", fontsize=8)
ax.set_xlabel("phụ trội chọn lọc (tính bằng epoch của cả tập)")
ax.set_ylabel("ICR tập lõi ÷ ICR chọn ngẫu nhiên")
ax.set_title("Tập lõi 50 phần trăm: lợi thế ICR co lại khi tính phụ trội")
ax.set_ylim(0.4, 2.0)
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

Với giả định của sách, phụ trội phải dưới 0.4 epoch của cả tập. Đây chính là bất đẳng thức chọn lọc
của chương, viết bằng FLOP: chi phí chọn phải nhỏ hơn phần huấn luyện mà tập con tiết kiệm. Nếu tập lõi
chỉ giữ được 55 phần trăm mức tăng, gần như mọi phụ trội đều xoá sạch lợi thế.

## 3. Lợi ích giảm dần, đo trên dữ liệu thật

Chương giả sử thông tin của một tập cỡ $D$ tăng theo $\log D$, còn chi phí tính toán tăng tuyến tính,
$C(D) = O_{\text{sample}} \cdot D$. Khi đó ICR biên xấp xỉ $1 / (O_{\text{sample}} \cdot D)$: mỗi lần
nhân đôi dữ liệu, mỗi FLOP thêm vào đem lại khoảng một nửa phần học so với trước. Sách nói rõ đây là
mô hình đơn giản hoá, không phải định luật.

Bạn sẽ đo một đường cong học tập thật. Một mạng 784-128-10 học FashionMNIST trên các tập con từ 200 tới
10,000 ảnh. Mọi cỡ tập dùng cùng 15 epoch, nên chi phí tỉ lệ đúng với $D$ như mô hình của chương.

In [ ]:
import torch.nn.functional as F
import torchvision

tap_hl = torchvision.datasets.FashionMNIST(DATA, train=True, download=True)
tap_kt = torchvision.datasets.FashionMNIST(DATA, train=False, download=True)
N_MAX, N_KT = 10_000, 5_000
X = tap_hl.data[:N_MAX].reshape(-1, 784).float() / 255
y = tap_hl.targets[:N_MAX].clone()
X_kt = tap_kt.data[:N_KT].reshape(-1, 784).float() / 255
y_kt = tap_kt.targets[:N_KT].clone()
TEN_LOP = ["áo thun", "quần dài", "áo len", "váy", "áo khoác", "xăng-đan", "áo sơ mi", "giày thể thao", "túi", "bốt"]
assert len(tap_hl.classes) == len(TEN_LOP)
print(f"{N_MAX:,} ảnh huấn luyện, {N_KT:,} ảnh kiểm tra, 10 lớp: {', '.join(TEN_LOP)}")


def mlp(an=128):
    return torch.nn.Sequential(torch.nn.Linear(784, an), torch.nn.ReLU(), torch.nn.Linear(an, 10))


def huan_luyen(X_, y_, so_epoch=15, batch=128, an=128, seed=SEED):
    """Vòng lặp quen thuộc. foreach=False: với vài tensor nhỏ, bản lặp từng tham số của Adam
    tránh được phụ trội gộp tensor trên một số CPU."""
    dat_hat_giong(seed)
    m = mlp(an)
    toi_uu = torch.optim.Adam(m.parameters(), lr=1e-3, foreach=False)
    for _ in range(so_epoch):
        thu_tu = torch.randperm(len(X_))
        for i in range(0, len(X_), batch):
            chon = thu_tu[i:i + batch]
            mat_mat = F.cross_entropy(m(X_[chon]), y_[chon])
            toi_uu.zero_grad()
            mat_mat.backward()
            toi_uu.step()
    return m


def do_chinh_xac(m):
    with torch.no_grad():
        return 100 * (m(X_kt).argmax(1) == y_kt).float().mean().item()


# Đếm FLOP theo đúng quy ước của chương: một bước huấn luyện bằng 3 lượt xuôi (24.6 / 8.2).
MAC_XUOI = 784 * 128 + 128 * 10
FLOP_MOI_MAU = 2 * MAC_XUOI * (FLOP_BUOC / FLOP_XUOI)
SO_EPOCH = 15
print(f"{MAC_XUOI:,} MAC mỗi lượt xuôi → {FLOP_MOI_MAU:,.0f} FLOP huấn luyện mỗi mẫu mỗi epoch")

### Dự đoán

Từ 1,000 lên 10,000 ảnh là gấp 10 lần dữ liệu và gấp 10 lần FLOP. Độ chính xác sẽ tăng nhiều hơn,
bằng, hay ít hơn mức tăng từ 200 lên 1,000 ảnh, vốn chỉ gấp 5 lần?

In [ ]:
CO_TAP = [200, 500, 1_000, 2_000, 5_000, 10_000]
tron = torch.randperm(N_MAX, generator=torch.Generator().manual_seed(SEED))  # tập lồng nhau: 200 ⊂ 500 ⊂ ...
t0 = time.perf_counter()
dung_sach = []
for n in CO_TAP:
    chon = tron[:n]
    dung_sach.append(do_chinh_xac(huan_luyen(X[chon], y[chon], SO_EPOCH)))
    do_tren_may(f"{n:>6,} ảnh: độ chính xác", dung_sach[-1], "%")
do_tren_may("thời gian cả đường cong", time.perf_counter() - t0, "s")
dung_sach = np.array(dung_sach)
D = np.array(CO_TAP, float)
C = FLOP_MOI_MAU * D * SO_EPOCH
do_tren_may("từ 200 lên 1,000 ảnh, độ chính xác tăng", dung_sach[2] - dung_sach[0], "điểm")
do_tren_may("từ 1,000 lên 10,000 ảnh, độ chính xác tăng", dung_sach[5] - dung_sach[2], "điểm")

### ICR biên và PPD

Đọc đường cong theo hai chỉ số.

* **ICR biên** giữa hai cỡ tập kề nhau: điểm độ chính xác thêm được, chia cho FLOP thêm vào.
* **PPD** (performance-per-data), chỉ số của phần khung đo lường trong chương:
  $\text{PPD}(n) = \big(\text{Acc}(n) - \text{Acc}(0)\big) / n$. Ở đây sổ tay **giả định**
  $\text{Acc}(0) = 10$ phần trăm, mức đoán mò giữa 10 lớp.

In [ ]:
ACC_0 = 10.0  # giả định: chưa có mẫu nào thì chỉ đoán mò
icr_bien = np.diff(dung_sach) / np.diff(C)
ppd = (dung_sach - ACC_0) / D
print(f"{'D':>7} {'độ chính xác':>13} {'PPD (điểm / 1,000 mẫu)':>24} {'ICR biên (điểm / GFLOP)':>25}")
for i, n in enumerate(CO_TAP):
    bien = f"{icr_bien[i - 1] * GFLOP:25.4f}" if i else f"{'':>25}"
    print(f"{n:>7,} {dung_sach[i]:>12.1f}% {1000 * ppd[i]:>24.2f} {bien}")

# Mô hình của chương, khớp vào điểm đo: Acc ≈ a + b·log D, ICR biên ∝ 1/D.
b, a = np.polyfit(np.log(D), dung_sach, 1)
D_giua = np.sqrt(D[1:] * D[:-1])
mo_hinh_icr = icr_bien[0] * D_giua[0] / D_giua

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
luoi = np.geomspace(D[0], D[-1], 100)
ax1.semilogx(D, dung_sach, "o-", color="C0", lw=2, label="đo trên máy này")
ax1.semilogx(luoi, a + b * np.log(luoi), "--", color="C1", label="khớp a + b·log D (mô hình của chương)")
ax1.set_xticks(D, [f"{n:,}" for n in CO_TAP])
ax1.set_xlabel("số ảnh huấn luyện (thang log)")
ax1.set_ylabel("độ chính xác trên tập kiểm tra (%)")
ax1.set_title("Đường cong học tập (đo trên máy này)")
ax1.legend(loc="lower right", fontsize=8)

ax2.loglog(D_giua, icr_bien * GFLOP, "o-", color="C0", lw=2, label="ICR biên, đo trên máy này")
ax2.loglog(D_giua, mo_hinh_icr * GFLOP, "--", color="C1", label="∝ 1/D, nối từ điểm đầu")
ax2.set_xticks(D_giua, [f"{n:,.0f}" for n in D_giua])
ax2.minorticks_off()
ax2.set_xlabel("cỡ tập ở giữa hai điểm đo (trung bình nhân, thang log)")
ax2.set_ylabel("điểm độ chính xác mỗi GFLOP (thang log)")
ax2.set_title("Mỗi FLOP thêm vào đem lại ít phần học hơn")
ax2.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

Hình bên trái thường gần một đường thẳng trên trục log, đúng hình dạng $\log D$ mà chương giả sử, và
có thể cong xuống ở đầu phải khi mô hình nhỏ này chạm trần của nó. Hình bên phải là cùng thông tin đó,
đọc theo FLOP: ICR biên giảm khi tập lớn lên, gần với độ dốc $1/D$, có khi còn dốc hơn. Hai điểm đo kề
nhau có thể lên xuống thất thường, vì mỗi điểm chỉ là một lần huấn luyện.

PPD giảm theo cùng cách: mẫu thứ 10,000 đáng ít hơn hẳn mẫu thứ 200. Đó là chỗ chương nói dữ liệu thành
**thuế dữ liệu**: thêm mẫu vẫn cộng thêm FLOP, nhưng phần học đem lại gần như không còn.

### Một mốc tham chiếu ở quy mô lớn: Chinchilla

Ở quy mô mô hình ngôn ngữ, chương trích kết quả Chinchilla: trong miền đã khớp, số token tối ưu tỉ lệ
với căn bậc hai của ngân sách tính toán, $D_{\text{opt}} \propto \sqrt{C}$. Đây là một mốc tham chiếu,
không phải điểm tối ưu phổ quát.

In [ ]:
theo_sach("token thêm khi nhân đôi tính toán (%)", 100 * (np.sqrt(2) - 1), sach=41.4,
          nguon="√C · 41.4 phần trăm · 20 token mỗi tham số")
for k in (2, 4, 10, 100):
    print(f"tính toán × {k:>3}: số token tối ưu × {np.sqrt(k):5.2f}, phần còn lại dồn vào tham số")

Cùng một ý với đường cong ở trên: khi tính toán tăng, dữ liệu tối ưu tăng chậm hơn tính toán. Một
ngân sách lớn mà chỉ có một kho dữ liệu cố định thì phần dư phải đi đâu đó, và chọn lọc là cách tiêu
phần dư ấy vào chất lượng thay vì vào số lượt đọc lại.

## 4. Hệ số chất lượng dữ liệu

Napkin math 1.2 hỏi: cần thêm bao nhiêu dữ liệu nhiễu để đạt cùng sai số mục tiêu như dữ liệu sạch?
Trong mô hình **minh hoạ** của chương, sai số với dữ liệu sạch giảm theo $1/D$, còn với dữ liệu nhiễu
giảm theo $1/\sqrt{D}$. Vậy $D_{\text{sạch}} \propto 1/\varepsilon$ và
$D_{\text{nhiễu}} \propto 1/\varepsilon^2$.

In [ ]:
EPS = sach(0.01, nguon="ε = 0.01 · 100 · 10,000 · 100×")
D_sach = 1 / EPS
D_nhieu = 1 / EPS ** 2
theo_sach("D sạch ở ε = 0.01", D_sach, sach=100, nguon="ε = 0.01 · 100 · 10,000 · 100×")
theo_sach("D nhiễu ở ε = 0.01", D_nhieu, sach=10_000, nguon="ε = 0.01 · 100 · 10,000 · 100×")
theo_sach("dữ liệu nhiễu cần gấp", D_nhieu / D_sach, sach=100, nguon="ε = 0.01 · 100 · 10,000 · 100×")

# Mô hình nhiễu đối xứng nhị phân: lật nhãn với tỉ lệ ρ thì cỡ mẫu hiệu dụng nhân với (1 − 2ρ)².
RHO = sach(10, nguon="ρ = 10 phần trăm") / 100
theo_sach("số mẫu cần thêm ở ρ = 10%", 1 / (1 - 2 * RHO) ** 2, sach=1.56,
          nguon="ρ = 10 phần trăm · (1 − 2ρ)² · ≈1.56×", tol=0.005)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
eps = np.geomspace(0.002, 0.2, 50)
ax1.loglog(eps, 1 / eps, color="C0", lw=2, label="sạch: D ∝ 1/ε")
ax1.loglog(eps, 1 / eps ** 2, color="C3", lw=2, label="nhiễu: D ∝ 1/ε²")
ax1.plot([EPS, EPS], [D_sach, D_nhieu], "o", color="0.2")
ax1.annotate("100", (EPS, D_sach), xytext=(8, -12), textcoords="offset points", fontsize=8)
ax1.annotate("10,000: gấp 100 lần", (EPS, D_nhieu), xytext=(8, 4), textcoords="offset points", fontsize=8)
ax1.invert_xaxis()
ax1.set_xlabel("sai số mục tiêu ε (thang log, càng sang phải càng khắt khe)")
ax1.set_ylabel("số mẫu cần (tỉ lệ, thang log)")
ax1.set_title("Napkin math 1.2 (mô hình minh hoạ)")
ax1.legend(loc="upper left", fontsize=8)

rho = np.linspace(0, 0.4, 81)
ax2.semilogy(rho * 100, 1 / (1 - 2 * rho) ** 2, color="C3", lw=2)
ax2.plot([RHO * 100], [1 / (1 - 2 * RHO) ** 2], "o", color="0.2")
ax2.annotate("ρ = 10 %: ≈1.56×", (RHO * 100, 1.5625), xytext=(10, 12), textcoords="offset points", fontsize=8)
ax2.set_yticks([1, 2, 5, 10, 20], ["1×", "2×", "5×", "10×", "20×"])
ax2.minorticks_off()
ax2.set_xlabel("tỉ lệ lật nhãn ρ (%)")
ax2.set_ylabel("số mẫu cần, so với dữ liệu sạch (thang log)")
ax2.set_title("Nhiễu đối xứng nhị phân: 1 / (1 − 2ρ)²")
plt.tight_layout()
plt.show()

Sách gọi kết luận "làm sạch dữ liệu hoạt động như một bộ tăng tốc tính toán 100 lần" là kết quả của
riêng các giả định ấy, và nói hệ số thật tuỳ tải công việc (workload). Phần còn lại đo hệ số đó cho một
tải cụ thể.

### Cấy nhiễu nhãn vào FashionMNIST

Lật nhãn của một phần ảnh huấn luyện sang một lớp khác, chọn ngẫu nhiên đều. Tập kiểm tra giữ nhãn đúng,
vì đó mới là tác vụ thật. Huấn luyện lại cả đường cong học tập ở hai mức nhiễu, 20 và 40 phần trăm.

**Dự đoán:** với 40 phần trăm nhãn sai, cần gấp bao nhiêu ảnh để bằng độ chính xác của 1,000 ảnh sạch?

In [ ]:
def cay_nhieu(y_, ti_le, seed=1):
    """Lật nhãn của một phần mẫu sang một lớp khác, chọn đều. Trả về nhãn mới và mặt nạ mẫu bị lật."""
    g = torch.Generator().manual_seed(seed)
    lat = torch.rand(len(y_), generator=g) < ti_le
    moi = y_.clone()
    moi[lat] = (moi[lat] + torch.randint(1, 10, (int(lat.sum()),), generator=g)) % 10
    return moi, lat


MUC_NHIEU = [0.2, 0.4]
dung_nhieu = {0.0: dung_sach}
for ti_le in MUC_NHIEU:
    y_nhieu, lat = cay_nhieu(y, ti_le)
    assert bool((y_nhieu[lat] != y[lat]).all()) and bool((y_nhieu[~lat] == y[~lat]).all())
    kq = []
    for n in CO_TAP:
        chon = tron[:n]
        kq.append(do_chinh_xac(huan_luyen(X[chon], y_nhieu[chon], SO_EPOCH)))
    dung_nhieu[ti_le] = np.array(kq)
    do_tren_may(f"nhiễu {ti_le:.0%}, độ chính xác ở 10,000 ảnh", kq[-1], "%")


def can_bao_nhieu(duong, muc):
    """Số ảnh để một đường cong chạm độ chính xác `muc`, nội suy theo log D; NaN nếu không chạm."""
    if muc > duong.max():
        return np.nan
    tot_nhat = np.maximum.accumulate(duong)  # đường cong đơn điệu để nội suy được
    return float(np.exp(np.interp(muc, tot_nhat, np.log(D))))


K = 10
print(f"{'mức nhiễu':>9} | " + " | ".join(f"bằng {n:,} ảnh sạch" for n in (500, 1_000, 2_000)) + " | mở rộng của sổ tay")
for ti_le in MUC_NHIEU:
    he_so = []
    for n in (500, 1_000, 2_000):
        can = can_bao_nhieu(dung_nhieu[ti_le], dung_sach[CO_TAP.index(n)])
        he_so.append(f"{can / n:>13.1f}×" if np.isfinite(can) else f"{'không chạm':>14}")
    mo_rong = 1 / (1 - ti_le * K / (K - 1)) ** 2
    print(f"{ti_le:>9.0%} | " + " | ".join(f"{h:>18}" for h in he_so) + f" | {mo_rong:>17.2f}×")

fig, ax = plt.subplots(figsize=(7.5, 4))
for (ti_le, duong), mau in zip(dung_nhieu.items(), ("C0", "C1", "C3")):
    ax.semilogx(D, duong, "o-", color=mau, lw=2, label=f"{ti_le:.0%} nhãn bị lật")
muc = dung_sach[CO_TAP.index(1_000)]
ax.axhline(muc, color="0.5", lw=0.8, ls=":")
ax.text(D[0], muc + 0.6, "mức của 1,000 ảnh sạch", fontsize=8, color="0.3")
ax.set_xticks(D, [f"{n:,}" for n in CO_TAP])
ax.set_xlabel("số ảnh huấn luyện (thang log)")
ax.set_ylabel("độ chính xác trên tập kiểm tra sạch (%)")
ax.set_title("Nhiễu nhãn dời cả đường cong học tập sang phải (đo trên máy này)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Đọc hình theo chiều ngang: khoảng cách từ đường sạch tới đường nhiễu, ở cùng một độ chính xác, là số
ảnh mà nhiễu bắt bạn trả thêm. Bảng in hệ số ấy ở vài mức. Cột cuối là một phép **mở rộng của sổ tay**,
không phải của sách: với 10 lớp và nhiễu đối xứng, hiệu giữa xác suất của nhãn đúng và của một nhãn sai
bất kỳ là $1 - \rho K/(K-1)$, thay cho $1 - 2\rho$ của trường hợp nhị phân. Hệ số đo được và hệ số của
công thức thường cùng bậc, nhưng không trùng: công thức bỏ qua việc mạng còn học được bao nhiêu từ chính
các nhãn sai.

Mô hình $1/\sqrt{D}$ của napkin math 1.2 là một kịch bản minh hoạ, không phải dự báo cho tải này, nên đừng
đặt hai hệ số cạnh nhau. Điều chương muốn bạn mang theo là hình dạng: nhiễu làm mỗi mẫu đáng ít hơn, nên
làm sạch có giá trị như mua thêm dữ liệu.

### Một mô hình thay thế có tìm lại được nhãn sai không?

Cắt tỉa theo chất lượng dùng độ tin cậy của mô hình để tìm nhãn có thể sai. Huấn luyện một mô hình thay
thế nhỏ, chỉ 2 epoch, trên 10,000 ảnh có 20 phần trăm nhãn bị lật, rồi xếp mọi mẫu theo mất mát với
chính nhãn của nó. Mẫu bị lật thường có mất mát cao, vì mô hình chưa kịp học thuộc chúng.

In [ ]:
y_nhieu, lat = cay_nhieu(y, 0.2)
proxy = huan_luyen(X, y_nhieu, so_epoch=2, an=32)
with torch.no_grad():
    mat_mat_mau = F.cross_entropy(proxy(X), y_nhieu, reduction="none")
nghi_ngo = torch.argsort(mat_mat_mau, descending=True)
so_lat = int(lat.sum())
print(f"{so_lat:,} nhãn bị lật trên {N_MAX:,} ảnh")
for k in (500, 1_000, so_lat):
    do_tren_may(f"trong {k:,} mẫu bị nghi nhất, phần thật sự bị lật", 100 * lat[nghi_ngo[:k]].float().mean().item(), "%")
print(f"so với chọn ngẫu nhiên: đúng {100 * so_lat / N_MAX:.0f} phần trăm")

# Hai cách dùng điểm số. Tự động xoá: bỏ đúng bằng số nhãn sai, những mẫu bị nghi nhất.
# Xem xét: một người duyệt 1,000 mẫu bị nghi nhất và chỉ sửa những nhãn thật sự sai.
xoa = nghi_ngo[:so_lat]
xoa_oan, con_sot = int((~lat[xoa]).sum()), int(lat[nghi_ngo[so_lat:]].sum())
do_tren_may("tự động xoá: mẫu nhãn đúng bị xoá oan", xoa_oan, "mẫu")
assert xoa_oan == con_sot  # xoá đúng bằng số nhãn sai, nên mỗi lần xoá oan là một nhãn sai bị sót
print(f"và cũng chừng ấy nhãn sai còn sót lại trong tập: {con_sot:,}")
DUYET = 1_000
tim_thay = int(lat[nghi_ngo[:DUYET]].sum())
ngau_nhien = int(lat[torch.randperm(N_MAX, generator=torch.Generator().manual_seed(SEED))[:DUYET]].sum())
do_tren_may(f"duyệt {DUYET:,} mẫu bị nghi nhất: nhãn sai tìm thấy", tim_thay, "mẫu")
do_tren_may(f"duyệt {DUYET:,} mẫu chọn ngẫu nhiên: nhãn sai tìm thấy", ngau_nhien, "mẫu")

In [ ]:
sai_nhung_nghi = [int(i) for i in nghi_ngo[:400] if not lat[i]][:6]
lat_va_nghi = [int(i) for i in nghi_ngo[:400] if lat[i]][:6]
fig, axes = plt.subplots(2, 6, figsize=(11, 5.4))
for hang, (ds, tieu_de) in enumerate(((lat_va_nghi, "nhãn bị lật"), (sai_nhung_nghi, "nhãn đúng\nmà vẫn bị nghi"))):
    for ax, i in zip(axes[hang], ds):
        ax.imshow(X[i].reshape(28, 28), cmap="gray_r")
        ax.set_title(f"ghi: {TEN_LOP[y_nhieu[i]]}\nthật: {TEN_LOP[y[i]]}", fontsize=7)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    axes[hang][0].set_ylabel(tieu_de, fontsize=9)
fig.suptitle("Những mẫu mô hình thay thế nghi nhất", fontsize=11)
plt.tight_layout(h_pad=2.0)
plt.show()

Phần đúng trong những mẫu bị nghi thường cao hơn hẳn mức ngẫu nhiên, nên một mô hình rẻ đã đủ để dồn
công xem xét vào đúng chỗ: cùng 1,000 lượt duyệt, danh sách của mô hình tìm ra nhiều nhãn sai hơn hẳn
một danh sách ngẫu nhiên. Nhưng hàng dưới cho thấy vì sao chương dặn **điểm số nên dẫn tới xem xét,
không tự động xoá**: một số mẫu bị nghi có nhãn hoàn toàn đúng, chỉ là chúng khó, thường thuộc hai lớp
trông giống nhau. Tự động xoá bỏ luôn những mẫu ấy, tức đúng những mẫu nằm gần biên quyết định, nơi mô
hình cần học nhất.

## Thử thêm

1. Ở phần 2, đặt `GIU = 0.3` (giữ 30 phần trăm) và tìm phần mức tăng tối thiểu để tập lõi vẫn hơn chọn
   ngẫu nhiên khi phụ trội bằng 0.2 epoch. (Các dấu ✗ khi đó là điều được chờ đợi.)
2. Ở phần 3, đổi `SO_EPOCH` thành 5. Đường cong học tập ở đầu trái thay đổi ra sao, và vì sao một tập nhỏ
   với quá ít bước cập nhật trông như "thiếu dữ liệu" trong khi thật ra là thiếu huấn luyện?
3. Ở phần 4, thay `cay_nhieu` bằng nhiễu **không đối xứng**: chỉ lật áo sơ mi (lớp 6) thành áo thun
   (lớp 0). Mô hình thay thế có còn tìm ra chúng dễ như trước không?

## Tóm lại

* ICR đặt phần học và FLOP vào một tỉ số. Trong napkin math 1.1, tập lõi 50 phần trăm có ICR cao hơn 1.8
  lần chỉ vì nó giữ 90 phần trăm mức tăng với 50 phần trăm dữ liệu, và lợi thế ấy biến mất khi phụ trội
  chọn lọc vượt 0.4 epoch.
* Đường cong học tập đo được có hình dạng chương mô tả: độ chính xác gần tuyến tính theo log D, nên ICR
  biên và PPD giảm khi tập lớn lên.
* Nhiễu nhãn dời đường cong sang phải, tức mỗi mẫu đáng ít hơn. Hệ số 100 lần của sách là của một mô hình
  minh hoạ; hệ số thật phải đo trên tải của mình.
* Một mô hình thay thế rẻ tìm được phần lớn nhãn sai, nhưng cũng nghi cả những mẫu khó mà đúng, nên điểm
  số là để xem xét, không phải để xoá tự động.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Chọn lọc dữ liệu — bớt việc trước khi việc bắt đầu](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch09-data-selection/), dựng từ chương
[*Data Selection*](https://mlsysbook.ai/vol1/data_selection/data_selection.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.